In [ ]:
# TODO: replace this with databricks

%load_ext sql
%sql duckdb:///:memory:

%sql create or replace view V_SERIATIM as select * from read_parquet('../../mort_data/V_SERIATIM.parquet')

import sys
sys.path.insert(0, "../../")

The 'toml' package isn't installed. To load settings from pyproject.toml or ~/.jupysql/config, install with: pip install toml

Connecting to 'duckdb:///:memory:'

Running query in 'duckdb:///:memory:'

# Individual Life Data Analysis

This notebook provides an examples of how to use the **scaffolding tools** located in the lib directory to analyze real individual life data.  The data is derived from the SOA's public-use dataset (ILEC).

In this exercise you'll learn how to use a simple and powerful model for data exploration called a **Decision Tree**.

## Decision Trees

When the question is *why is the A/E or lift high or low?*, **Decision Trees** are the first tool you should reach for.  They provide:

 * Highly explainable outputs.

 * The ability to test different theories / variables as the "best" explanation.

 * Very powerful assistance with refinining GLMs.  Especially identifying missing interactions.


In [ ]:
%%sql

select 
    OBSERVATION_YEAR,
    sum(CLAIM_COUNT)::int as CLAIM_COUNT,
    sum(EXPECTED_COUNT)::int AS EXPECTED_COUNT,
    round(sum(CLAIM_COUNT) / sum(EXPECTED_COUNT), 2) AS AE_RATIO
from V_SERIATIM 
group by OBSERVATION_YEAR
order by OBSERVATION_YEAR

Running query in 'duckdb:///:memory:'

OBSERVATION_YEAR,CLAIM_COUNT,EXPECTED_COUNT,AE_RATIO
2009,6366,6016,1.06
2010,4737,4548,1.04
2011,9121,8518,1.07
2012,5338,4850,1.1
2013,5779,5287,1.09
2014,6548,5980,1.1
2015,7404,6912,1.07
2016,8349,8096,1.03
2017,9496,9000,1.06
2018,8230,8283,0.99


### Building a decision tree

The **scaffolding library** makes building an interpretable decision tree very easy.

In [ ]:
from src.lib.tree import PoissonDecisionTree


pdt = PoissonDecisionTree(
    actual_col="CLAIM_COUNT",
    offset_col="EXPECTED_COUNT"     # another word for the expected basis is an "offset" 
)

df_all_data = %sql select * from V_SERIATIM
df_all_data = df_all_data.DataFrame()

# don't forget to remove EXPOSURE or any other columns that are not 
# good explanations for A/E drivers (e.g. policy_id, etc.)
df_tree_data = df_all_data.drop(columns=["GROUP_ID", "OBS_ID", "POLICIES_EXPOSED", "FACE_AMOUNT_GRP_ORD"])

pdt.fit(df_tree_data)

pdt.print_tree()

Running query in 'duckdb:///:memory:'

root: ae=1.0575   y=71368
  - PREFERRED_CLASS_2 < 0.5: ae=0.9144   y=24335
    - PREFERRED_CLASS_3 < 0.5: ae=0.8239   y=18069
      - ATTAINED_AGE < 82.5: ae=0.7960   y=14215
      - ATTAINED_AGE >= 82.5: ae=0.9464   y=3854
    - PREFERRED_CLASS_3 >= 0.5: ae=1.3379   y=6266
      - ATTAINED_AGE < 54.5: ae=1.7011   y=1480
      - ATTAINED_AGE >= 54.5: ae=1.2551   y=4786
  - PREFERRED_CLASS_2 >= 0.5: ae=1.1506   y=47033
    - DURATION < 1.5: ae=2.6336   y=417
      - NUMBER_OF_PREFERRED_CLASSES_3 < 0.5: ae=3.4051   y=362
      - NUMBER_OF_PREFERRED_CLASSES_3 >= 0.5: ae=1.0572   y=55
    - DURATION >= 1.5: ae=1.1448   y=46616
      - ATTAINED_AGE < 56.5: ae=1.4104   y=4609
      - ATTAINED_AGE >= 56.5: ae=1.1216   y=42007


#### Understanding the output

``` root: ae=1.0575   y=71368 ```  -> Overall A/E = 1.0575%, there are 71,368 claims in this dataset (`y`)

**The most important variable is Preferred Class #2** since it occurs in the first level below the root.  

`  - PREFERRED_CLASS_2 < 0.5: ae=0.9144   y=24335`  <- first variable below root = most important

`    - ...`

`  - PREFERRED_CLASS_2 >= 0.5: ae=1.1506   y=47033`  <- ALSO part of the first variable below root = most important

`    - ...`

`PREFERRED_CLASS_2` is a catagorical variable that the decision tree has changed to a [one-hot encoding](https://www.datacamp.com/tutorial/one-hot-encoding-python-tutorial). Compare the above decision tree output  with the below table (made with SQL):

In [20]:
%%sql

select 
    case 
        when PREFERRED_CLASS = '2' then 'PREFERRED_CLASS = 2'
        else 'PREFERRED_CLASS_2 != 2'
    end as VARIABLE,
    sum(CLAIM_COUNT)::int as CLAIM_COUNT,
    sum(EXPECTED_COUNT)::int AS EXPECTED_COUNT,
    round(sum(CLAIM_COUNT) / sum(EXPECTED_COUNT), 2) AS AE_RATIO
from V_SERIATIM 
group by
    case 
        when PREFERRED_CLASS = '2' then 'PREFERRED_CLASS = 2'
        else 'PREFERRED_CLASS_2 != 2'
    end 
order by round(sum(CLAIM_COUNT) / sum(EXPECTED_COUNT), 2)


Running query in 'duckdb:///:memory:'

VARIABLE,CLAIM_COUNT,EXPECTED_COUNT,AE_RATIO
PREFERRED_CLASS_2 != 2,24335,26613,0.91
PREFERRED_CLASS = 2,47033,40877,1.15


#### Decision Tree Example Wrap-up

 * Decision trees are very good at **finding which variables explain A/E differences**.  They are credibility aware, so they will generally find credible splits.

 * **They are good at finding pockets of good / bad experience (data)**.  Don't stop there though!  Dig in, and look for other alternative answers!  In this example, Preferred #2 may be the pocket that's performing worst, but that's not the whole answer.

In short, decision trees help us find columns (variables) and rows (pockets of experience) to look at deeper.

#### Why decision trees work well for actuarial data

Decision trees address the **shortcomings of A/E ratios**.  

 * An A/E of 200% is meaningless unless we know the number of claims.  A 200% A/E is fine when 1 claim was expected, and we observed 2.  But a 200% A/E is __not__ fine when we see 500 claims vs. 250 expected.

 * We need at least one more number to interpret an A/E, either the expected or actual claims.

*Which of these deviations is the worst / most impactful?*

| Actual | Expected | A/E  | 
|--------|----------|------|
| 200    | 150      | 133% |
| 7,500  | 7,000    | 107% |

The top number is off by 50 claims, and the bottom is off by 500.  But the top number looks "worse" according to A/E.  Rather than using A/E, we need a measurement that **accounts for credibility**.  We can use the Poisson distribution to accomplish this.

Recreate the below table in Excel.  Copy the below formula into a fourth column called *Negative Log Likelihood (NLL)* .

```
=-1 * LOG(POISSON.DIST(A2,B2,FALSE))
```

| Actual | Expected | A/E  |  NLL | 
|--------|----------|------|-----------------|
| 200    | 150      | 133% |  4.82          |
| 7,500  | 7,000    | 107% |  9.91          |

Note that the larger NLL corresponds to the second row.  According to the Poisson distribution, this row has the more significant A/E difference.

 * Compare small and large value for actual and expected.  Note that the NLL reflects the credibility in actual and expected AND the difference between them.

### Key Takeaways

In simplest terms, **decision trees use NLL, rather than the A/E ratio** to find the largest errors in the predictions. It focuses on explaining the second row (with a 107% A/E) before looking at the 133% row, because of the much higher credibility.

Use them because they are **highly interpretable**. 

Use them for:

 1. Quick explorations of data to find the **most** predictive variables.
 
 2. Refining GLMs by adding interactions, or tweaking knots (for splines).
